# The agent loop: a hands-on implementation

**Track:** Agents · **Guide:** [`052-agent-loop.md`](../guides/concepts/052-agent-loop.md)

The agent loop repeatedly observes context, chooses an action, receives a result, and stops at a limit or completion.

> **Intuition:** A technician checks a gauge, takes one permitted action, checks the new reading, and stops when the job is done.

This lesson builds a small working example in stages: prepare inputs, implement
the core idea, run it on a realistic toy case, and inspect its output and limits.
It runs offline unless a cell is explicitly marked as an optional provider call.

```mermaid
flowchart LR
    A[Prepare inputs] --> B[Implement The agent loop]
    B --> C[Run on example]
    C --> D[Evaluate and inspect]
```

**Case study:** A troubleshooting assistant gathers one diagnostic signal per step and hands off rather than looping forever.

Run the cells from top to bottom. To install the notebook environment, follow
[`guides/SETUP.md`](../guides/SETUP.md). The toy data and implementation are for
learning, not a production quality, safety, or security guarantee.


## 1. Define bounded state and tools

An agent loop observes state, selects one permitted action, and stops on completion or a step budget.


In [ ]:
state = {"goal": "find_return_window", "evidence": [], "status": "running"}
knowledge = {
    "returns": "Returns are accepted within 30 days.",
    "support": "Support is available on weekdays.",
}
tools = {"search": lambda key: knowledge.get(key, "No evidence found.")}
max_steps = 4

## 2. Run observe–act–observe cycles

This deterministic policy is executable without an LLM. A model can propose actions, but the application should still validate every action.


In [ ]:
def choose_action(state):
    return "search:returns" if not state["evidence"] else "finish"

trace = []
for step_number in range(1, max_steps + 1):
    action = choose_action(state)
    trace.append({"step": step_number, "action": action})
    if action == "finish":
        state["status"] = "completed"
        break
    tool_name, argument = action.split(":", maxsplit=1)
    if tool_name not in tools:
        raise PermissionError(tool_name)
    result = tools[tool_name](argument)
    state["evidence"].append(result)
else:
    state["status"] = "limit_reached"

## 3. Inspect the trace and stop condition

A production loop needs timeouts, tool-call limits, input validation, and human approval before consequential actions.


In [ ]:
print("trace:", trace)
print("final state:", state)
assert state["status"] == "completed"

## Wrap-up and next steps

- Identify what was learned, stored, retrieved, or generated in this implementation.
- Change an input and observe how the output changes.
- Test one failure case and decide when the system should abstain or ask for help.
- Replace toy data/components only after you can evaluate the change against a
  held-out or representative test set.

Continue with the [concept guide](../guides/concepts/052-agent-loop.md) and the
[learning roadmap](../guides/ROADMAP.md).
